In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import StratifiedShuffleSplit, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.utils import shuffle
from sklearn.metrics import confusion_matrix
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

# Load your dataset
data = pd.read_csv('2yr.txt', delimiter=',')
data1 = pd.read_excel('Michael_Model_RUN-6-7.xlsx', sheet_name='2Lines-fake_Training_data')
data = data.append(data1)
data['TM_STAMP'] = (pd.to_datetime(data['TM_STAMP']).astype('int64') // 10**9)

data=data.dropna()


val = pd.read_csv('Q1_2023 (1).txt', delimiter=',')
val['TM_STAMP'] = (pd.to_datetime(val['TM_STAMP']).astype('int64') // 10**9)
val=val.dropna()

In [ ]:
data

In [ ]:
val

In [ ]:
# Preprocess the data
X = data[['ID_PIEZO', 'TM_STAMP', 'DIGITS','AMP']]
y = data['REPORTABLE']
X, y = shuffle(X, y, random_state=42)


In [ ]:
validation_data = (X_val, y_val)

In [ ]:
# Standardize the features
scaler = StandardScaler()
X_train = scaler.fit_transform(X)
#X_val = scaler.transform(x_val)
#X_test = scaler.transform(x_test)

In [ ]:
val_y = val['REPORTABLE'].astype('float')
val_x = val[['ID_PIEZO', 'TM_STAMP', 'DIGITS','AMP']]

val_x = scaler.transform(val_x)


In [ ]:
def get_model():
    model = keras.Sequential([
         layers.Dense(128, activation="relu"),
         layers.Dense(128, activation="relu"),
         layers.Dense(1, activation="sigmoid")
    ])
    model.compile(optimizer="rmsprop",
                  loss="binary_crossentropy",
                  metrics=["accuracy"])
    return model

model = get_model()
history_noise = model.fit(
    X_train, y,
    epochs = 10,
    batch_size=512
    )                             #validation_data = (X_val, y_val)

In [ ]:
history_dict = history_noise.history
history_dict.keys()

In [ ]:
history_dict = history_noise.history
loss_values = history_dict["loss"]
val_loss_values = history_dict["val_loss"]
epochs = range(1, len(loss_values) + 1)
plt.plot(epochs, loss_values, "bo", label="Training loss")
plt.plot(epochs, val_loss_values, "b", label="Validation loss")
plt.title("Training and validation loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()
plt.show()

In [ ]:
plt.clf()
acc = history_dict["accuracy"]
val_acc = history_dict["val_accuracy"]
plt.plot(epochs, acc, "bo", label="Training acc")
plt.plot(epochs, val_acc, "b", label="Validation acc")
plt.title("Training and validation accuracy")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend()
plt.show()

In [ ]:
#results = model.evaluate(X_test, y_test)

In [ ]:
#pred = model.predict(X_test)
#pred

In [ ]:
for i in range(len(pred)):
    if pred[i] >= 0.5:
        pred[i] = 1
    else:
        pred[i] = 0

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
cm = confusion_matrix(y_test, pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("Confusion Matrix")
plt.xlabel("Predicted Labels")
plt.ylabel("True Labels")
plt.show()

In [ ]:
pred_val = model.evaluate(val_x, val_y)

In [ ]:
pre_val= model.predict(val_x)
pre_val

In [ ]:
for i in range(len(pre_val)):
    if pre_val[i] >= 0.5:
        pre_val[i] = 1
    else:
        pre_val[i] = 0

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
cm = confusion_matrix(val_y, pre_val)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("Confusion Matrix")
plt.xlabel("Predicted Labels")
plt.ylabel("True Labels")
plt.show()

In [ ]:
# Add the predictions to the new dataset
val['PREDICTIONS'] = pre_val

val['TM_STAMP'] = pd.to_datetime(val['TM_STAMP'], unit='s')
val

In [ ]:
import pandas as pd
from sklearn.metrics import confusion_matrix

# Assuming you already have the new_data dataframe
# Step 2: Create a new column to store the comparison results
val['COMPARE'] = val['REPORTABLE'] == (val['PREDICTIONS'] == 1)

# Get unique ID_PIEZO values for predicted false and actual REPORTABLE false
predicted_false = val.loc[val['PREDICTIONS'] == 0, 'ID_PIEZO'].unique()
actual_reportable_false = val.loc[val['REPORTABLE'] == False, 'ID_PIEZO'].unique()

# Save the unique ID_PIEZO's for predicted false and actual REPORTABLE false in separate text files
with open("PIEZO_Predicted_False.txt", 'w') as f:
    for id_piezo in predicted_false:
        f.write(str(id_piezo) + '\n')

with open("Actual_Piezo_Reportable_False.txt", 'w') as f:
    for id_piezo in actual_reportable_false:
        f.write(str(id_piezo) + '\n')

# Step 4: Create a confusion matrix
y_true = val['REPORTABLE']
y_pred = val['PREDICTIONS'].apply(lambda x: x == 1)  # Convert 1's and 0's to boolean values
cm = confusion_matrix(y_true, y_pred)

# Create a labeled confusion matrix using a Pandas DataFrame
labeled_cm = pd.DataFrame(cm, index=['Actual: False', 'Actual: True'], columns=['Predicted: False', 'Predicted: True'])

print("Confusion Matrix:")
print(labeled_cm)

In [ ]:
val['REPORTABLE'] =val['REPORTABLE'].astype('float')

In [ ]:
val

In [ ]:
reportable = val['REPORTABLE'].to_numpy()

In [ ]:
predict = val['PREDICTIONS'].to_numpy()
predict

In [ ]:
ids = val['ID_PIEZO'].to_numpy()
amps = val['AMP'].to_numpy()
digits = val['DIGITS'].to_numpy()

In [ ]:
plt.clf()
ID = []                
AMPS = []
DIGITS = []
FALSE_PREDICTIONS = []
for i in  range(len(reportable)):
    if reportable[i] != predict[i]:
        ID.append(ids[i])
        AMPS.append(amps[i])
        DIGITS.append(digits[i])
        FALSE_PREDICTIONS.append(predict[i])  
        

In [ ]:
#plt.plot(ID, AMPS, "b", label="AMPS")
#plt.plot(ID, DIGITS, 'r', label='DIGITS')
#plt.plot(FALSE_PREDICTIONS, 'bo', label='FALSE_PREDICTIONS')
plt.xlabel("PIEZO_IDs")
plt.legend()
plt.show()
            

In [ ]:
# Create figure and axis objects
fig, ax1 = plt.subplots()

# Plot data on the first axis
ax1.plot(ID, AMPS, 'go', markersize=3, label='AMPS')
ax1.set_xlabel('PZ_ID')
ax1.set_ylabel('AMPS', color='g')
ax1.tick_params('y', colors='g')

# Create a second axis sharing the same x-axis
ax2 = ax1.twinx()

# Plot data on the second axis
ax2.plot(ID, DIGITS, 'bo', markersize=3, label='DIGITS')
ax2.set_ylabel('DIGITS', color='b')
ax2.tick_params('y', colors='b')

# Create a third axis sharing the same x-axis
ax3 = ax1.twinx()

# Offset the third axis
ax3.spines['right'].set_position(('outward', 60))

# Plot data on the third axis
ax3.plot(ID, FALSE_PREDICTIONS, 'rx', markersize=3, label='FALSE_PREDICTIONS')
ax3.set_ylabel('FALSE_PREDICTIONS', color='r')
ax3.tick_params('y', colors='r')

# Add a legend
#lines = ax1.get_lines() + ax2.get_lines() + ax3.get_lines()
#x1.legend(lines, [line.get_label() for line in lines])

# Show the plot
plt.show()

In [ ]:
len(set(ID))